Дорогой студент!

В домашнем задании Lite вам предлагается поработать подробнее с параметрами словаря и формированием гиперпараметров нейронной сети. Создайте 9 нейросетей с различными гиперпараметрами (см. пунтк 2 и 3)

 Для этого необходимо:

  1. Воссоздать ноутбук, аналогичный ноутбуку практической части №1, загрузив при этом необходимую нам базу (код уже доступен в ноутбуке).

  2. Задать в ноутбуке следующие параметры для размера словаря, ширины окна и шага:

    - Размер словаря - от 10000 до 20000 (выбрать меньшее значение диапазона, если будет перегрузка ОЗУ и перезапуск подключения к Colaboratory)
    - Ширина окна - от 1000 до 2000
    - Шаг - от 100 до 500 (на обучение лучше влияет наименьший шаг, но это может перегрузить ОЗУ).

  3. Создать архитектуру сети и задать гиперпараметры. Можно воспользоваться шаблоном:
  
   - Добавьте модель прямого распространения **Sequential()**
   - Добавьте один или несколько полносвязных (**Dense**) слоёв
   - Добавьте слои **Dropout()** и **BatchNormalization()**
   - Добавьте выходной полносвязный слой с количеством нейронов, соответствующим количеству классов (число писателей)
  
   Напомним, что точность сети можно проверить по значению показателя 'val_accuracy' на конце каждой эпохи.
   

## 1. Импорт библиотек

In [36]:
# Импорт библиотек
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import gdown
import os
import re
import matplotlib.pyplot as plt

## 2. Загрузка и распаковка датасета

In [ ]:
# Загрузка и распаковка датасета
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l7/writers.zip', None, quiet=True)
!unzip -qo writers.zip -d writers/

Датасет загружен и распакован


## 3. Чтение текстов писателей и формирование списка классов

In [38]:
# Чтение и подготовка текстов
FILE_DIR = 'writers'
SIG_TRAIN = 'обучающая'
SIG_TEST = 'тестовая'

CLASS_LIST = []
text_train = []
text_test = []

for file_name in os.listdir(FILE_DIR):
    m = re.match('\((.+)\) (\S+)_', file_name)
    if m:
        class_name = m[1]
        subset_name = m[2].lower()
        is_train = SIG_TRAIN in subset_name
        is_test = SIG_TEST in subset_name
        if is_train or is_test:
            if class_name not in CLASS_LIST:
                CLASS_LIST.append(class_name)
                text_train.append('')
                text_test.append('')
            cls = CLASS_LIST.index(class_name)
            with open(f'{FILE_DIR}/{file_name}', 'r') as f:
                text = f.read()
            subset = text_train if is_train else text_test
            subset[cls] += ' ' + text.replace('\n', ' ')

CLASS_COUNT = len(CLASS_LIST)
VOCAB_SIZE = 15000
print(f"Классы: {CLASS_LIST}")
print(f"Количество классов: {CLASS_COUNT}")

Классы: ['Макс Фрай', 'О. Генри', 'Рэй Брэдберри', 'Клиффорд_Саймак', 'Стругацкие', 'Булгаков']
Количество классов: 6


<>:11: SyntaxWarning: invalid escape sequence '\('
<>:11: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_480/11208369.py:11: SyntaxWarning: invalid escape sequence '\('
  m = re.match('\((.+)\) (\S+)_', file_name)


## 4. Токенизация текстов и преобразование в последовательности индексов

In [ ]:
# Токенизация и построение словаря
tokenizer = Tokenizer(num_words=VOCAB_SIZE,
                      filters='!"#$%&()*+,-–—./…:;<=>?@[\\]^_`{|}~«»\t\n\xa0\ufeff',
                      lower=True, split=' ', oov_token='неизвестное_слово')
tokenizer.fit_on_texts(text_train)

seq_train = tokenizer.texts_to_sequences(text_train)
seq_test = tokenizer.texts_to_sequences(text_test)

Токенизация завершена


## 5. Функции разбиения на фрагменты и создания модели

In [40]:
# Функция разбиения последовательности на отрезки
def split_sequence(sequence, win_size, hop):
    if len(sequence) < win_size:
        return []
    return [sequence[i:i + win_size] for i in range(0, len(sequence) - win_size + 1, hop)]

# Функция формирования выборок
def vectorize_sequence(seq_list, win_size, hop):
    x_list = []
    y_list = []
    for cls, seq in enumerate(seq_list):
        chunks = split_sequence(seq, win_size, hop)
        for ch in chunks:
            x_list.append(ch)
            y_list.append(cls)
    x_pad = pad_sequences(x_list, maxlen=win_size, padding='post', truncating='post')
    return torch.tensor(x_pad, dtype=torch.long), torch.tensor(y_list, dtype=torch.long)

## 6. Создание модели

In [69]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, win_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1 = nn.Linear(win_size * embed_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.3)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.embedding(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        return self.fc3(x)

## 7. Функция обучения модели 

In [70]:
# Функция обучения модели
def train_model(model, train_loader, val_loader, epochs=10, lr=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    val_accs = []

    for epoch in range(epochs):
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()

        scheduler.step()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                outputs = model(x_batch)
                _, predicted = torch.max(outputs, 1)
                total += y_batch.size(0)
                correct += (predicted == y_batch).sum().item()

        val_acc = correct / total
        val_accs.append(val_acc)
        print(f"Epoch {epoch+1}/{epochs} - val_accuracy: {val_acc:.4f}")

    return max(val_accs)

## 8. Эксперимент 1: WIN_SIZE=1000, WIN_HOP=100

In [71]:
# Эксперимент 1: WIN_SIZE=1000, HOP=100
print("\n=== ЭКСПЕРИМЕНТ 1: WIN_SIZE=1000, HOP=100 ===")

WIN_SIZE = 1000
WIN_HOP = 100

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=128, shuffle=False)

model = TextClassifier(VOCAB_SIZE, 128, CLASS_COUNT, WIN_SIZE)
acc1 = compile_and_train(model, train_loader, test_loader, epochs=10)

print(f"Результат 1: Лучшая val_accuracy = {acc1:.4f}")


=== ЭКСПЕРИМЕНТ 1: WIN_SIZE=1000, HOP=100 ===
Epoch 1/10 - val_accuracy: 0.3889
Epoch 2/10 - val_accuracy: 0.4777
Epoch 3/10 - val_accuracy: 0.5021
Epoch 4/10 - val_accuracy: 0.4874
Epoch 5/10 - val_accuracy: 0.5036
Epoch 6/10 - val_accuracy: 0.5115
Epoch 7/10 - val_accuracy: 0.5165
Epoch 8/10 - val_accuracy: 0.5121
Epoch 9/10 - val_accuracy: 0.5070
Epoch 10/10 - val_accuracy: 0.5132
Результат 1: Лучшая val_accuracy = 0.5132


## 9. Эксперимент 2: WIN_SIZE=1000, WIN_HOP=300

In [72]:
# Эксперимент 2: WIN_SIZE=1000, HOP=300
print("\n=== ЭКСПЕРИМЕНТ 2: WIN_SIZE=1000, HOP=300 ===")

WIN_SIZE = 1000
WIN_HOP = 300

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=128, shuffle=False)

model = TextClassifier(VOCAB_SIZE, 128, CLASS_COUNT, WIN_SIZE)
acc2 = compile_and_train(model, train_loader, test_loader, epochs=10)

print(f"Результат 2: Лучшая val_accuracy = {acc2:.4f}")


=== ЭКСПЕРИМЕНТ 2: WIN_SIZE=1000, HOP=300 ===
Epoch 1/10 - val_accuracy: 0.3478
Epoch 2/10 - val_accuracy: 0.3411
Epoch 3/10 - val_accuracy: 0.3572
Epoch 4/10 - val_accuracy: 0.3550
Epoch 5/10 - val_accuracy: 0.3720
Epoch 6/10 - val_accuracy: 0.3707
Epoch 7/10 - val_accuracy: 0.3702
Epoch 8/10 - val_accuracy: 0.3684
Epoch 9/10 - val_accuracy: 0.3805
Epoch 10/10 - val_accuracy: 0.3734
Результат 2: Лучшая val_accuracy = 0.3734


## 10. Эксперимент 3: WIN_SIZE=1000, WIN_HOP=500

In [73]:
# Эксперимент 3: WIN_SIZE=1000, HOP=500
print("\n=== ЭКСПЕРИМЕНТ 3: WIN_SIZE=1000, HOP=500 ===")

WIN_SIZE = 1000
WIN_HOP = 500

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=128, shuffle=False)

model = TextClassifier(VOCAB_SIZE, 128, CLASS_COUNT, WIN_SIZE)
acc3 = compile_and_train(model, train_loader, test_loader, epochs=10)

print(f"Результат 3: Лучшая val_accuracy = {acc3:.4f}")


=== ЭКСПЕРИМЕНТ 3: WIN_SIZE=1000, HOP=500 ===
Epoch 1/10 - val_accuracy: 0.3022
Epoch 2/10 - val_accuracy: 0.3306
Epoch 3/10 - val_accuracy: 0.3194
Epoch 4/10 - val_accuracy: 0.3097
Epoch 5/10 - val_accuracy: 0.3194
Epoch 6/10 - val_accuracy: 0.3201
Epoch 7/10 - val_accuracy: 0.3351
Epoch 8/10 - val_accuracy: 0.3187
Epoch 9/10 - val_accuracy: 0.3284
Epoch 10/10 - val_accuracy: 0.3291
Результат 3: Лучшая val_accuracy = 0.3291


## 11. Эксперимент 4: WIN_SIZE=1500, WIN_HOP=100

In [74]:
# Эксперимент 4: WIN_SIZE=1500, HOP=100
print("\n=== ЭКСПЕРИМЕНТ 4: WIN_SIZE=1500, HOP=100 ===")

WIN_SIZE = 1500
WIN_HOP = 100

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=128, shuffle=False)

model = TextClassifier(VOCAB_SIZE, 128, CLASS_COUNT, WIN_SIZE)
acc4 = compile_and_train(model, train_loader, test_loader, epochs=10)

print(f"Результат 4: Лучшая val_accuracy = {acc4:.4f}")


=== ЭКСПЕРИМЕНТ 4: WIN_SIZE=1500, HOP=100 ===
Epoch 1/10 - val_accuracy: 0.4093
Epoch 2/10 - val_accuracy: 0.5165
Epoch 3/10 - val_accuracy: 0.5219
Epoch 4/10 - val_accuracy: 0.5370
Epoch 5/10 - val_accuracy: 0.5406
Epoch 6/10 - val_accuracy: 0.5452
Epoch 7/10 - val_accuracy: 0.5409
Epoch 8/10 - val_accuracy: 0.5473
Epoch 9/10 - val_accuracy: 0.5377
Epoch 10/10 - val_accuracy: 0.5466
Результат 4: Лучшая val_accuracy = 0.5466


## 12. Эксперимент 5: WIN_SIZE=1500, WIN_HOP=300

In [75]:
# Эксперимент 5: WIN_SIZE=1500, HOP=300
print("\n=== ЭКСПЕРИМЕНТ 5: WIN_SIZE=1500, HOP=300 ===")

WIN_SIZE = 1500
WIN_HOP = 300

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=128, shuffle=False)

model = TextClassifier(VOCAB_SIZE, 128, CLASS_COUNT, WIN_SIZE)
acc5 = compile_and_train(model, train_loader, test_loader, epochs=10)

print(f"Результат 5: Лучшая val_accuracy = {acc5:.4f}")


=== ЭКСПЕРИМЕНТ 5: WIN_SIZE=1500, HOP=300 ===
Epoch 1/10 - val_accuracy: 0.3306
Epoch 2/10 - val_accuracy: 0.3568
Epoch 3/10 - val_accuracy: 0.3680
Epoch 4/10 - val_accuracy: 0.3806
Epoch 5/10 - val_accuracy: 0.3860
Epoch 6/10 - val_accuracy: 0.3928
Epoch 7/10 - val_accuracy: 0.3883
Epoch 8/10 - val_accuracy: 0.3842
Epoch 9/10 - val_accuracy: 0.3892
Epoch 10/10 - val_accuracy: 0.3941
Результат 5: Лучшая val_accuracy = 0.3941


## 13. Эксперимент 6: WIN_SIZE=1500, WIN_HOP=500

In [76]:
# Эксперимент 6: WIN_SIZE=1500, HOP=500
print("\n=== ЭКСПЕРИМЕНТ 6: WIN_SIZE=1500, HOP=500 ===")

WIN_SIZE = 1500
WIN_HOP = 500

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=128, shuffle=False)

model = TextClassifier(VOCAB_SIZE, 128, CLASS_COUNT, WIN_SIZE)
acc6 = compile_and_train(model, train_loader, test_loader, epochs=10)

print(f"Результат 6: Лучшая val_accuracy = {acc6:.4f}")


=== ЭКСПЕРИМЕНТ 6: WIN_SIZE=1500, HOP=500 ===
Epoch 1/10 - val_accuracy: 0.2856
Epoch 2/10 - val_accuracy: 0.3411
Epoch 3/10 - val_accuracy: 0.3508
Epoch 4/10 - val_accuracy: 0.3673
Epoch 5/10 - val_accuracy: 0.3598
Epoch 6/10 - val_accuracy: 0.3523
Epoch 7/10 - val_accuracy: 0.3463
Epoch 8/10 - val_accuracy: 0.3658
Epoch 9/10 - val_accuracy: 0.3673
Epoch 10/10 - val_accuracy: 0.3508
Результат 6: Лучшая val_accuracy = 0.3508


## 14. Эксперимент 7: WIN_SIZE=2000, WIN_HOP=100

In [77]:
# Эксперимент 7: WIN_SIZE=2000, HOP=100
print("\n=== ЭКСПЕРИМЕНТ 7: WIN_SIZE=2000, HOP=100 ===")

WIN_SIZE = 2000
WIN_HOP = 100

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=128, shuffle=False)

model = TextClassifier(VOCAB_SIZE, 128, CLASS_COUNT, WIN_SIZE)
acc7 = compile_and_train(model, train_loader, test_loader, epochs=10)

print(f"Результат 7: Лучшая val_accuracy = {acc7:.4f}")


=== ЭКСПЕРИМЕНТ 7: WIN_SIZE=2000, HOP=100 ===
Epoch 1/10 - val_accuracy: 0.4940
Epoch 2/10 - val_accuracy: 0.5477
Epoch 3/10 - val_accuracy: 0.5478
Epoch 4/10 - val_accuracy: 0.5676
Epoch 5/10 - val_accuracy: 0.5528
Epoch 6/10 - val_accuracy: 0.5555
Epoch 7/10 - val_accuracy: 0.5605
Epoch 8/10 - val_accuracy: 0.5626
Epoch 9/10 - val_accuracy: 0.5557
Epoch 10/10 - val_accuracy: 0.5696
Результат 7: Лучшая val_accuracy = 0.5696


## 15. Эксперимент 8: WIN_SIZE=2000, WIN_HOP=300

In [78]:
# Эксперимент 8: WIN_SIZE=2000, HOP=300
print("\n=== ЭКСПЕРИМЕНТ 8: WIN_SIZE=2000, HOP=300 ===")

WIN_SIZE = 2000
WIN_HOP = 300

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=128, shuffle=False)

model = TextClassifier(VOCAB_SIZE, 128, CLASS_COUNT, WIN_SIZE)
acc8 = compile_and_train(model, train_loader, test_loader, epochs=10)

print(f"Результат 8: Лучшая val_accuracy = {acc8:.4f}")


=== ЭКСПЕРИМЕНТ 8: WIN_SIZE=2000, HOP=300 ===
Epoch 1/10 - val_accuracy: 0.3668
Epoch 2/10 - val_accuracy: 0.3650
Epoch 3/10 - val_accuracy: 0.4016
Epoch 4/10 - val_accuracy: 0.3962
Epoch 5/10 - val_accuracy: 0.4003
Epoch 6/10 - val_accuracy: 0.4034
Epoch 7/10 - val_accuracy: 0.4116
Epoch 8/10 - val_accuracy: 0.3980
Epoch 9/10 - val_accuracy: 0.4034
Epoch 10/10 - val_accuracy: 0.4066
Результат 8: Лучшая val_accuracy = 0.4066


## 16. Эксперимент 9: WIN_SIZE=2000, WIN_HOP=500

In [79]:
# Эксперимент 9: WIN_SIZE=2000, HOP=500
print("\n=== ЭКСПЕРИМЕНТ 9: WIN_SIZE=2000, HOP=500 ===")

WIN_SIZE = 2000
WIN_HOP = 500

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=128, shuffle=False)

model = TextClassifier(VOCAB_SIZE, 128, CLASS_COUNT, WIN_SIZE)
acc9 = compile_and_train(model, train_loader, test_loader, epochs=10)

print(f"Результат 9: Лучшая val_accuracy = {acc9:.4f}")


=== ЭКСПЕРИМЕНТ 9: WIN_SIZE=2000, HOP=500 ===
Epoch 1/10 - val_accuracy: 0.3268
Epoch 2/10 - val_accuracy: 0.3720
Epoch 3/10 - val_accuracy: 0.3788
Epoch 4/10 - val_accuracy: 0.3803
Epoch 5/10 - val_accuracy: 0.3765
Epoch 6/10 - val_accuracy: 0.3810
Epoch 7/10 - val_accuracy: 0.3886
Epoch 8/10 - val_accuracy: 0.3916
Epoch 9/10 - val_accuracy: 0.3976
Epoch 10/10 - val_accuracy: 0.3938
Результат 9: Лучшая val_accuracy = 0.3938


## 17. Сравнение результатов 9 экспериментов

In [80]:
# Сравнение результатов
print("\nЭксперимент | WIN_SIZE | WIN_HOP | Лучшая val_accuracy")
print("------------------------------------------------------")
print(f"     1      |   1000   |   100   |      {acc1:.4f}")
print(f"     2      |   1000   |   300   |      {acc2:.4f}")
print(f"     3      |   1000   |   500   |      {acc3:.4f}")
print(f"     4      |   1500   |   100   |      {acc4:.4f}")
print(f"     5      |   1500   |   300   |      {acc5:.4f}")
print(f"     6      |   1500   |   500   |      {acc6:.4f}")
print(f"     7      |   2000   |   100   |      {acc7:.4f}")
print(f"     8      |   2000   |   300   |      {acc8:.4f}")
print(f"     9      |   2000   |   500   |      {acc9:.4f}")

best_acc = max([acc1, acc2, acc3, acc4, acc5, acc6, acc7, acc8, acc9])
print(f"\nЛучший результат: {best_acc:.4f} ({best_acc*100:.2f}%)")

if best_acc == acc1: print("Лучшие параметры: WIN_SIZE=1000, WIN_HOP=100")
elif best_acc == acc2: print("Лучшие параметры: WIN_SIZE=1000, WIN_HOP=300")
elif best_acc == acc3: print("Лучшие параметры: WIN_SIZE=1000, WIN_HOP=500")
elif best_acc == acc4: print("Лучшие параметры: WIN_SIZE=1500, WIN_HOP=100")
elif best_acc == acc5: print("Лучшие параметры: WIN_SIZE=1500, WIN_HOP=300")
elif best_acc == acc6: print("Лучшие параметры: WIN_SIZE=1500, WIN_HOP=500")
elif best_acc == acc7: print("Лучшие параметры: WIN_SIZE=2000, WIN_HOP=100")
elif best_acc == acc8: print("Лучшие параметры: WIN_SIZE=2000, WIN_HOP=300")
elif best_acc == acc9: print("Лучшие параметры: WIN_SIZE=2000, WIN_HOP=500")


Эксперимент | WIN_SIZE | WIN_HOP | Лучшая val_accuracy
------------------------------------------------------
     1      |   1000   |   100   |      0.5132
     2      |   1000   |   300   |      0.3734
     3      |   1000   |   500   |      0.3291
     4      |   1500   |   100   |      0.5466
     5      |   1500   |   300   |      0.3941
     6      |   1500   |   500   |      0.3508
     7      |   2000   |   100   |      0.5696
     8      |   2000   |   300   |      0.4066
     9      |   2000   |   500   |      0.3938

Лучший результат: 0.5696 (56.96%)
Лучшие параметры: WIN_SIZE=2000, WIN_HOP=100
